# Script 10b — Fine-tuning DistilGPT-2 on my Letterboxd reviews

We fine-tune a pre-trained language model on my translated reviews.
The model learns my writing style and can generate new reviews.

**Model:** DistilGPT-2 (82M parameters, runs well on CPU)  
**Data:** ~417 reviews translated to English  
**Goal:** given a film title and rating, generate a review in my style


## 1. Imports

In [6]:
import os, sys, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import torch
from transformers import (
    GPT2Tokenizer, GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer, TrainingArguments,
    pipeline
)
from datasets import Dataset

SCRIPT_DIR = os.path.join(os.getcwd(), '..')
DATA_DIR   = os.path.join(SCRIPT_DIR, 'data', 'processed')
OUT_DIR    = os.path.join(SCRIPT_DIR, 'output')
MODEL_DIR  = os.path.join(SCRIPT_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")


Device: cpu
PyTorch version: 2.11.0+cpu


## 2. Load and prepare the reviews

We format each review as a structured prompt:

```
<|film|> The Dark Knight (2008) <|rating|> 5.0 <|review|> ... <|endoftext|>
```

This teaches the model to associate film context with writing style.


In [7]:
df = pd.read_csv(os.path.join(DATA_DIR, 'reviews_translated.csv'))
df = df[df['translated_review'].notna()].copy()
df['translated_review'] = df['translated_review'].str.strip()
df = df[df['translated_review'].str.len() > 30]  # skip very short reviews

print(f"Reviews available: {len(df)}")
print(f"Avg length: {df['translated_review'].str.len().mean():.0f} chars")
print(f"Min length: {df['translated_review'].str.len().min()} chars")
print(f"Max length: {df['translated_review'].str.len().max()} chars")
print()

# Format each review as a structured prompt
def format_review(row):
    rating = f"{float(row['Rating']):.1f}" if pd.notna(row['Rating']) else "?"
    return (
        f"<|film|> {row['Name']} ({int(row['Year'])}) "
        f"<|rating|> {rating} "
        f"<|review|> {row['translated_review']} "
        f"<|endoftext|>"
    )

df['formatted'] = df.apply(format_review, axis=1)

print("Sample formatted review:")
print(df['formatted'].iloc[0][:300])


Reviews available: 414
Avg length: 618 chars
Min length: 32 chars
Max length: 2337 chars

Sample formatted review:
<|film|> Soul (2020) <|rating|> 4.5 <|review|> One of the most impactful films I've seen in recent times. And what I cried most about Pixar. Above all, it adds a much more sensitive, much more inspiring touch to an animated film. Something I haven't seen in a genre like this for a long time. <|endof


## 3. Load DistilGPT-2 tokenizer and model

**DistilGPT-2** is a distilled (compressed) version of GPT-2.
It's 40% smaller and 60% faster while retaining most of the quality.
Perfect for fine-tuning on CPU with a small dataset.


In [8]:
print("Loading DistilGPT-2...")

tokenizer = GPT2Tokenizer.from_pretrained('distilgpt2')

# Add our special tokens so the model learns the structure
special_tokens = {
    'pad_token': '<|pad|>',
    'additional_special_tokens': ['<|film|>', '<|rating|>', '<|review|>']
}
tokenizer.add_special_tokens(special_tokens)

model = GPT2LMHeadModel.from_pretrained('distilgpt2')
model.resize_token_embeddings(len(tokenizer))  # resize for new tokens

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {total_params/1e6:.0f}M parameters")
print(f"Vocabulary size: {len(tokenizer)}")


Loading DistilGPT-2...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Model loaded: 82M parameters
Vocabulary size: 50261


## 4. Tokenise the dataset

We convert all text to token IDs (numbers) that the model understands.
We also split into train (90%) and validation (10%).


In [9]:
# Tokenise all reviews
def tokenise(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=256,        # max tokens per review
        padding='max_length',
    )

texts = df['formatted'].tolist()

# Train / validation split
split = int(len(texts) * 0.9)
train_texts = texts[:split]
val_texts   = texts[split:]

train_dataset = Dataset.from_dict({'text': train_texts}).map(tokenise, batched=True)
val_dataset   = Dataset.from_dict({'text': val_texts}).map(tokenise, batched=True)

# Set labels = input_ids for language modelling (predict next token)
train_dataset = train_dataset.map(lambda x: {'labels': x['input_ids']})
val_dataset   = val_dataset.map(lambda x: {'labels': x['input_ids']})

print(f"Training samples:   {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")


Map:   0%|          | 0/372 [00:00<?, ? examples/s]

Map:   0%|          | 0/42 [00:00<?, ? examples/s]

Map:   0%|          | 0/372 [00:00<?, ? examples/s]

Map:   0%|          | 0/42 [00:00<?, ? examples/s]

Training samples:   372
Validation samples: 42


## 5. Fine-tune the model

We train for 5 epochs — the model sees the full dataset 5 times.

**What is an epoch?**  
One complete pass through all training reviews.
More epochs = more learning, but risk of overfitting (memorising instead of learning).

This will take **20-40 minutes on CPU**. The loss should decrease each epoch.


In [11]:
training_args = TrainingArguments(
    output_dir=os.path.join(MODEL_DIR, 'checkpoints'),
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir=os.path.join(MODEL_DIR, 'logs'),
    logging_steps=20,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='none',
    use_cpu=True,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

trainer.train()


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss
1,3.603352,3.150985
2,3.032680,3.004109
3,2.908968,2.996808
4,2.730599,3.006648
5,2.734169,3.009175


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=465, training_loss=3.0742622949743783, metrics={'train_runtime': 3688.9104, 'train_samples_per_second': 0.504, 'train_steps_per_second': 0.126, 'total_flos': 121502989025280.0, 'train_loss': 3.0742622949743783, 'epoch': 5.0})

## 6. Save the model

In [12]:
model_path = os.path.join(MODEL_DIR, 'letterboxd_gpt2')
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)
print(f"Model saved to: {model_path}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: c:\Users\Huawei\Desktop\Private\Cinemaverse\scripts\..\models\letterboxd_gpt2


## 7. Generate reviews!

Now the fun part — give the model a film title and rating,
and let it write a review in your style.


In [13]:
def generate_review(film_name, year, rating, max_length=200, temperature=0.8):
    """
    Generate a review for a film using the fine-tuned model.
    
    temperature: controls randomness
      - low (0.5) = more conservative, repetitive
      - high (1.2) = more creative, sometimes incoherent
    """
    prompt = f"<|film|> {film_name} ({year}) <|rating|> {rating:.1f} <|review|>"
    
    inputs = tokenizer.encode(prompt, return_tensors='pt')
    
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_length=max_length,
            temperature=temperature,
            do_sample=True,
            top_k=50,           # only consider top 50 tokens
            top_p=0.95,         # nucleus sampling
            repetition_penalty=1.2,  # penalise repeating phrases
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    generated = tokenizer.decode(outputs[0], skip_special_tokens=False)
    
    # Extract just the review part
    if '<|review|>' in generated:
        review = generated.split('<|review|>')[1]
        review = review.replace('<|endoftext|>', '').strip()
        return review
    return generated

# Test with a few films
test_films = [
    ("Oppenheimer", 2023, 5.0),
    ("Madame Web",  2024, 0.5),
    ("Dune: Part Two", 2024, 5.0),
]

for film, year, rating in test_films:
    print(f"Film: {film} ({year}) — Rating: {rating}")
    print("-" * 50)
    review = generate_review(film, year, rating)
    print(review)
    print()


Film: Oppenheimer (2023) — Rating: 5.0
--------------------------------------------------
I don't know if that's a good review, but for me it feels solid even more than usual...
It’s hard to describe the film with this rating… The truth is you're not going anywhere and never thinking about what happened in "A24". But once again they delivered an excellent performance: cinematography was flawless - incredible at times; one of those scenes where everything just goes according as planned — well done together throughout all their phases—all culminating from start to finish."

Film: Madame Web (2024) — Rating: 0.5
--------------------------------------------------
It's like watching a movie from the beginning of your life and seeing everything they had to learn about it all becomes even more so when you realize that this project is literally written for them, in addition to being filmed at home… I'm not sure if there were any spoilers or plot points added during those days — but honestly: w

## 8. Batch generate for watchlist recommendations

Generate reviews for the top recommended films from script 08b.


In [14]:
import json

# Load top watchlist recommendations
try:
    recs = pd.read_csv(os.path.join(DATA_DIR, 'recs_watchlist.csv')).head(10)
    
    generated_reviews = []
    for _, row in recs.iterrows():
        film   = row['Name']
        year   = int(row['Year'])
        # Use predicted rating from script 09 if available, else similarity score
        rating = float(row.get('predicted_rating', 4.0))
        
        print(f"Generating: {film} ({year}) — {rating:.1f}*")
        review = generate_review(film, year, rating, temperature=0.85)
        generated_reviews.append({
            'Name':             film,
            'Year':             year,
            'predicted_rating': rating,
            'generated_review': review,
        })
        print(f"  {review[:120]}...")
        print()
    
    # Save
    pd.DataFrame(generated_reviews).to_csv(
        os.path.join(DATA_DIR, 'generated_reviews.csv'),
        index=False, encoding='utf-8'
    )
    print("Generated reviews saved to data/processed/generated_reviews.csv")

except FileNotFoundError:
    print("Run script 08b first to generate watchlist recommendations.")
    print("Then re-run this cell.")


Generating: The Good, the Bad and the Ugly (1966) — 4.0*
  I'm so thankful to be a part of that project… Now imagine how much fun your movie has already experienced in this film.....

Generating: Persona (1966) — 4.0*
  “A film with an extremely dramatic premise, filled with deeply unsettling plot twists…
I’m still not sure what the hell...

Generating: A Patch of Blue (1965) — 4.0*
  This is really a work in progress, and I can honestly say it has some pretty solid ideas — especially the final scenes w...

Generating: Serpico (1973) — 4.0*
  I couldn’t handle it better than this, but a lot of people asked me why they watched the film…
It was so simple—well do...

Generating: High and Low (1963) — 4.0*
  I was intrigued by all the characters, but this one just made me stop looking for a project like that: It's incredibly w...

Generating: Kes (1969) — 4.0*
  Another year or so ago, I got a call from someone with my own personal opinion of the film: this is one helluva project....

Gene

## 9. Experiment — adjust temperature

Try different temperatures to see how they affect the output.
Lower = more like your writing. Higher = more creative but less coherent.


In [15]:
film, year, rating = "The Substance", 2024, 4.0

print(f"Film: {film} ({year}) — Rating: {rating}")
print()

for temp in [0.5, 0.8, 1.1]:
    print(f"Temperature = {temp}:")
    print("-" * 40)
    review = generate_review(film, year, rating, temperature=temp)
    print(review[:200])
    print()


Film: The Substance (2024) — Rating: 4.0

Temperature = 0.5:
----------------------------------------
I'm not going to lie: this is a film about the drug war and it's genuinely disturbing, but at least one of them has some positives that make me want more…
I don't know what else would have said if th

Temperature = 0.8:
----------------------------------------
I don’t want to be the only one in this genre, but it's kind of simple: if someone is listening and they feel that way — what does he actually mean?
This project has absolutely everything going for y

Temperature = 1.1:
----------------------------------------
It felt like I might have been born with autism, or something — a disorder that in itself made me want to do anything even worse… but on an overall level it was actually quite enjoyable and at least o



## Summary

**What we built:**
- Fine-tuned DistilGPT-2 on 417 Letterboxd reviews
- Model learns writing style, tone and vocabulary
- Can generate new reviews given a film title + rating

**Limitations:**
- 417 reviews is a small dataset — the model captures style patterns but isn't perfect
- Sometimes repeats phrases or loses coherence mid-sentence
- Works best for films/genres similar to what's in the training data

**The model is saved in `models/letterboxd_gpt2/`** — it can be loaded and used in the final dashboard.
